In [3]:
!pip install faiss-cpu sentence-transformers
import json
import faiss
import numpy as np
import re
from sentence_transformers import SentenceTransformer

# ==========================================
# CONFIGURATION & PATHS
# ==========================================

# Using the granular chunks file for high-accuracy RAG
RAG_CHUNKS_PATH = "rag_chunks.jsonl"

# The Superset handles strict numerical/boolean logic
SUPERSET_PATH = "MASTER_STRUCTURED_SUPERSET_2026-1.jsonl"

# Metadata containing real plan names and source_urls for citations
URL_METADATA_PATH = "metadata (1).json"

INDEX_PATH = "faiss_multi_provider_index.bin"
CHUNK_METADATA_PATH = "chunk_metadata.json"

# ==========================================
# 1. LOAD MODELS
# ==========================================

print("Loading embedding model...")
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

# ==========================================
# 2. LOAD METADATA (For URLs and Human Names)
# ==========================================

print("Mapping metadata for URLs and citations...")
url_lookup = {}
human_plan_names = {}
plan_summaries = {}

try:
    with open(URL_METADATA_PATH, "r") as f:
        meta_data = json.load(f)
        for item in meta_data:
            doc_id = item.get("doc_id", "").lower()
            if not doc_id:
                continue

            url_lookup[doc_id] = item.get("source_url", "No URL Available")
            human_plan_names[doc_id] = item.get("plan_name", "Unknown Plan")
            plan_summaries[doc_id] = item.get("text", "")

    print(f"Loaded {len(url_lookup)} URLs from metadata.")
except FileNotFoundError:
    print(f"Warning: {URL_METADATA_PATH} not found. Citations will be missing.")

def get_plan_details(doc_id):
    """Retrieves human name and URL based on doc_id."""
    doc_id_lower = doc_id.lower()
    return (
        human_plan_names.get(doc_id_lower, doc_id.title()),
        url_lookup.get(doc_id_lower, "No URL Available"),
        plan_summaries.get(doc_id_lower, "")
    )

# ==========================================
# 3. LOAD RAG CHUNKS (For Semantic Search)
# ==========================================

documents = []
chunk_metadata = []

print(f"Loading granular RAG chunks from {RAG_CHUNKS_PATH}...")
try:
    with open(RAG_CHUNKS_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)

            doc_id = obj.get("doc_id", "")
            provider = obj.get("insurer", "Unknown Provider")
            plan_name = obj.get("plan_name", "")
            chunk_text = obj.get("text", "")
            page_number = obj.get("page_start", "N/A")

            # Enrich the text sent to the embedder for better context
            enriched_chunk = (
                f"Provider: {provider}. Plan: {plan_name}. Page: {page_number}. "
                f"Document text: {chunk_text}"
            )

            documents.append(enriched_chunk)
            chunk_metadata.append({
                "provider": provider,
                "doc_id": doc_id,
                "plan_name": plan_name,
                "page_number": page_number,
                "chunk_text": chunk_text
            })
    print(f"Loaded {len(documents)} chunks.")
except FileNotFoundError:
    print(f"Error: {RAG_CHUNKS_PATH} not found.")

# ==========================================
# 4. LOAD STRUCTURED SUPERSET (Math & Logic)
# ==========================================

structured_data = {}

print("Loading structured data for numeric filtering...")
try:
    with open(SUPERSET_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)
            plan_key = obj.get("plan_name", "").lower()
            structured_data.setdefault(plan_key, []).append(obj)

    print(f"Loaded structured logic for {len(structured_data)} plans.")
except FileNotFoundError:
    print(f"Warning: {SUPERSET_PATH} not found. Structured scoring disabled.")

# ==========================================
# 5. BUILD FAISS INDEX
# ==========================================

print("Generating embeddings (this may take a moment)...")
if documents:
    embeddings = model.encode(documents, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    faiss.write_index(index, INDEX_PATH)

    with open(CHUNK_METADATA_PATH, "w") as f:
        json.dump(chunk_metadata, f)

    print("FAISS index built and saved successfully.")
else:
    print("No documents to index.")

# ==========================================
# 6. ADVANCED SCORING & FILTERING LOGIC
# ==========================================

def compute_structured_score(plan_key, query):
    plan_key_lower = plan_key.lower()

    matching_key = None
    for key in structured_data.keys():
        if key in plan_key_lower or plan_key_lower in key:
            matching_key = key
            break

    if not matching_key:
        return 0.5, True

    plan_entries = structured_data[matching_key]

    # HARD FILTERING
    query_lower = query.lower()
    if "maternity" in query_lower and not any(e.get("maternity_flag") for e in plan_entries):
        return 0.0, False
    if "psychiatric" in query_lower and not any(e.get("psychiatric_flag") for e in plan_entries):
        return 0.0, False
    if "fertility" in query_lower and not any(e.get("fertility_flag") for e in plan_entries):
        return 0.0, False

    # QUANTITATIVE SCORING
    excess_values = [float(e["excess_amount"]) for e in plan_entries if e.get("excess_amount") is not None]
    caps = [float(e["limit_amount"]) for e in plan_entries if e.get("limit_amount") is not None]

    avg_excess = np.mean(excess_values) if excess_values else 0
    avg_cap = np.mean(caps) if caps else 0

    excess_score = 1 / (1 + avg_excess / 250)
    cap_score = min(avg_cap / 50000, 1)

    structured_score = (0.7 * excess_score) + (0.3 * cap_score)
    return structured_score, True

# ==========================================
# 7. SEARCH PIPELINE
# ==========================================

def search_insurance(query, k=50):
    prefixed_query = "Represent this sentence for searching relevant passages: " + query
    query_embedding = model.encode([prefixed_query], normalize_embeddings=True)
    query_embedding = np.array(query_embedding).astype("float32")

    scores, indices = index.search(query_embedding, k)

    plan_chunks = {}
    provider_map = {}
    doc_id_map = {}

    # 1. Group ALL retrieved chunks by plan
    for score, idx in zip(scores[0], indices[0]):
        doc_id = chunk_metadata[idx]["doc_id"]
        plan_name = chunk_metadata[idx]["plan_name"]
        chunk_text = chunk_metadata[idx]["chunk_text"]

        if plan_name not in plan_chunks:
            plan_chunks[plan_name] = []
            provider_map[plan_name] = chunk_metadata[idx]["provider"]
            doc_id_map[plan_name] = doc_id

        plan_chunks[plan_name].append({
            "text": chunk_text,
            "score": float(score)
        })

    # 2. Detect explicit plan requests
    query_lower = query.lower()
    mentioned_plan = None

    for known_plan in plan_chunks.keys():
        clean_known_plan = known_plan.replace("(Table of Cover)", "").strip().lower()
        if clean_known_plan and clean_known_plan in query_lower:
            mentioned_plan = known_plan
            print(f"🎯 Detected explicit request for plan: {mentioned_plan}")
            break

    # 3. Query Rewriting for isolated plan searches
    if mentioned_plan:
        replace_target = mentioned_plan.replace("(Table of Cover)", "").strip()
        semantic_query = re.sub(replace_target, "", query, flags=re.IGNORECASE)

        # Re-embed the cleaned query
        clean_embedding = model.encode(["Represent this sentence for searching relevant passages: " + semantic_query], normalize_embeddings=True)
        clean_embedding = np.array(clean_embedding).astype("float32")

        # Re-score chunks locally
        for chunk in plan_chunks[mentioned_plan]:
            chunk_emb = model.encode([chunk["text"]], normalize_embeddings=True)
            chunk_emb = np.array(chunk_emb).astype("float32")
            chunk["score"] = float(np.dot(clean_embedding, chunk_emb.T)[0][0])

    results = []

    for plan_name, chunks in plan_chunks.items():
        # EXPLICIT PLAN FILTER
        if mentioned_plan and plan_name != mentioned_plan:
            continue

        doc_id = doc_id_map[plan_name]

        # Sort chunks by semantic score
        chunks = sorted(chunks, key=lambda x: x["score"], reverse=True)
        best_rag_score = chunks[0]["score"]

        # Compute Math Score
        structured_score, passes_filter = compute_structured_score(plan_name, query)
        if not passes_filter:
            continue

        normalized_rag = best_rag_score / (best_rag_score + 5)
        final_score = (normalized_rag * 0.7) + (structured_score * 0.3)

        human_name, source_url, summary = get_plan_details(doc_id)
        if human_name == doc_id.title():
            human_name = plan_name

        # Grab the top 2 chunks
        combined_evidence = "\n\n--- NEXT CHUNK ---\n\n".join(
            [c["text"] for c in chunks[:2]]
        )

        results.append({
            "plan_name": human_name,
            "provider": provider_map[plan_name],
            "source_url": source_url,
            "final_score": round(final_score, 4),
            "rag_score": round(normalized_rag, 4),
            "structured_score": round(structured_score, 4),
            "evidence": combined_evidence
        })

    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results[:5]

# ==========================================
# 8. EXECUTION / TEST
# ==========================================

if __name__ == "__main__":
    test_query = "What is the excess for maternity cover on the First Cover plan?"

    print(f"\nSearching for: '{test_query}'\n")
    if 'index' in locals():
        top_plans = search_insurance(test_query)

        for i, r in enumerate(top_plans, 1):
            print(f"Rank {i}: {r['plan_name']} ({r['provider']})")
            print(f"URL: {r['source_url']}")
            print(f"Scores -> Final: {r['final_score']} | Semantic: {r['rag_score']} | Quantitative: {r['structured_score']}")

            # Printing the full evidence block!
            print(f"\nEvidence Preview:\n{r['evidence']}\n")
            print("-" * 60)
    else:
        print("Cannot run search test; index was not built successfully.")

ModuleNotFoundError: No module named 'faiss'

In [ ]:
! pip install rank-bm25

In [ ]:
import json
import numpy as np
import re
from rank_bm25 import BM25Okapi

# ==========================================
# CONFIGURATION & PATHS
# ==========================================

RAG_CHUNKS_PATH = "rag_chunks.jsonl"
SUPERSET_PATH = "MASTER_STRUCTURED_SUPERSET_2026-1.jsonl"
URL_METADATA_PATH = "metadata (1).json"

# ==========================================
# 1. HELPER: TOKENIZER FOR BM25
# ==========================================
def tokenize(text):
    """Simple tokenizer that lowers text and extracts words/numbers."""
    return re.findall(r'\w+', text.lower())

# ==========================================
# 2. LOAD METADATA
# ==========================================
print("Mapping metadata for URLs and citations...")
url_lookup = {}
human_plan_names = {}

try:
    with open(URL_METADATA_PATH, "r") as f:
        meta_data = json.load(f)
        for item in meta_data:
            doc_id = item.get("doc_id", "").lower()
            if doc_id:
                url_lookup[doc_id] = item.get("source_url", "No URL Available")
                human_plan_names[doc_id] = item.get("plan_name", "Unknown Plan")
except FileNotFoundError:
    print(f"Warning: {URL_METADATA_PATH} not found.")

def get_plan_details(doc_id):
    doc_id_lower = doc_id.lower()
    return (human_plan_names.get(doc_id_lower, doc_id.title()), url_lookup.get(doc_id_lower, "No URL Available"))

# ==========================================
# 3. LOAD RAG CHUNKS
# ==========================================
documents = []
chunk_metadata = []

print(f"Loading granular RAG chunks from {RAG_CHUNKS_PATH}...")
try:
    with open(RAG_CHUNKS_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)

            doc_id = obj.get("doc_id", "")
            provider = obj.get("insurer", "Unknown Provider")
            plan_name = obj.get("plan_name", "")
            chunk_text = obj.get("text", "")

            # For BM25, passing just the raw text is usually best so we don't skew frequencies
            documents.append(chunk_text)

            chunk_metadata.append({
                "provider": provider,
                "doc_id": doc_id,
                "plan_name": plan_name,
                "chunk_text": chunk_text
            })
    print(f"Loaded {len(documents)} chunks.")
except FileNotFoundError:
    print(f"Error: {RAG_CHUNKS_PATH} not found.")

# ==========================================
# 4. LOAD STRUCTURED SUPERSET
# ==========================================
structured_data = {}

print("Loading structured data for numeric filtering...")
try:
    with open(SUPERSET_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)
            plan_key = obj.get("plan_name", "").lower()
            structured_data.setdefault(plan_key, []).append(obj)
except FileNotFoundError:
    print(f"Warning: {SUPERSET_PATH} not found.")

# ==========================================
# 5. BUILD BM25 INDEX
# ==========================================
print("Tokenizing documents and building BM25 index...")
tokenized_corpus = [tokenize(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 Index built successfully.")

# ==========================================
# 6. ADVANCED SCORING & FILTERING LOGIC
# ==========================================
def compute_structured_score(plan_key, query):
    plan_key_lower = plan_key.lower()

    matching_key = next((key for key in structured_data.keys() if key in plan_key_lower or plan_key_lower in key), None)
    if not matching_key:
        return 0.5, True

    plan_entries = structured_data[matching_key]
    query_lower = query.lower()

    # Hard Filters
    if "maternity" in query_lower and not any(e.get("maternity_flag") for e in plan_entries): return 0.0, False
    if "psychiatric" in query_lower and not any(e.get("psychiatric_flag") for e in plan_entries): return 0.0, False
    if "fertility" in query_lower and not any(e.get("fertility_flag") for e in plan_entries): return 0.0, False

    excess_values = [float(e["excess_amount"]) for e in plan_entries if e.get("excess_amount") is not None]
    avg_excess = np.mean(excess_values) if excess_values else 0
    excess_score = 1 / (1 + avg_excess / 250)

    return (0.7 * excess_score) + 0.15, True

# ==========================================
# 7. SEARCH PIPELINE (Using BM25)
# ==========================================
def search_insurance(query, k=50):
    query_lower = query.lower()
    mentioned_plan = None

    # 1. Detect explicit plan requests from available metadata
    unique_plans = set(meta["plan_name"] for meta in chunk_metadata)
    for known_plan in unique_plans:
        clean_known_plan = known_plan.replace("(Table of Cover)", "").strip().lower()
        if clean_known_plan and clean_known_plan in query_lower:
            mentioned_plan = known_plan
            print(f"🎯 Detected explicit request for plan: {mentioned_plan}")
            break

    # 2. Query Rewriting for BM25
    search_query = query
    if mentioned_plan:
        # Strip the plan name out so BM25 keyword matching focuses solely on the question
        replace_target = mentioned_plan.replace("(Table of Cover)", "").strip()
        search_query = re.sub(replace_target, "", search_query, flags=re.IGNORECASE)

    # 3. Get BM25 Scores
    tokenized_query = tokenize(search_query)
    doc_scores = bm25.get_scores(tokenized_query)

    # Get top k indices
    top_indices = np.argsort(doc_scores)[::-1][:k]

    plan_chunks = {}
    provider_map = {}
    doc_id_map = {}

    # 4. Group Top Results by Plan
    for idx in top_indices:
        score = doc_scores[idx]
        if score == 0: continue # Skip documents with zero matching keywords

        meta = chunk_metadata[idx]
        plan_name = meta["plan_name"]

        if plan_name not in plan_chunks:
            plan_chunks[plan_name] = []
            provider_map[plan_name] = meta["provider"]
            doc_id_map[plan_name] = meta["doc_id"]

        plan_chunks[plan_name].append({
            "text": meta["chunk_text"],
            "score": float(score)
        })

    results = []

    for plan_name, chunks in plan_chunks.items():
        # EXPLICIT PLAN FILTER
        if mentioned_plan and plan_name != mentioned_plan:
            continue

        doc_id = doc_id_map[plan_name]

        # Sort chunks by BM25 score
        chunks = sorted(chunks, key=lambda x: x["score"], reverse=True)
        best_bm25_score = chunks[0]["score"]

        # Compute Math Score
        structured_score, passes_filter = compute_structured_score(plan_name, query)
        if not passes_filter:
            continue

        # Normalize BM25 score (BM25 scores can be large, e.g., 10-40)
        normalized_bm25 = best_bm25_score / (best_bm25_score + 10)
        final_score = (normalized_bm25 * 0.7) + (structured_score * 0.3)

        human_name, source_url = get_plan_details(doc_id)
        if human_name == doc_id.title():
            human_name = plan_name

        combined_evidence = "\n\n--- NEXT CHUNK ---\n\n".join([c["text"] for c in chunks[:2]])

        results.append({
            "plan_name": human_name,
            "provider": provider_map[plan_name],
            "source_url": source_url,
            "final_score": round(final_score, 4),
            "rag_score": round(best_bm25_score, 4), # Showing RAW score for debugging
            "structured_score": round(structured_score, 4),
            "evidence": combined_evidence
        })

    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results[:5]

# ==========================================
# 8. EXECUTION / TEST
# ==========================================
if __name__ == "__main__":
    test_query = "What is the excess for maternity cover on the First Cover plan?"

    print(f"\nSearching for: '{test_query}'\n")
    top_plans = search_insurance(test_query)

    for i, r in enumerate(top_plans, 1):
        print(f"Rank {i}: {r['plan_name']} ({r['provider']})")
        print(f"URL: {r['source_url']}")
        print(f"Scores -> Final: {r['final_score']} | Raw BM25: {r['rag_score']} | Quantitative: {r['structured_score']}")
        print(f"\nEvidence Preview:\n{r['evidence']}\n")
        print("-" * 60)

Mapping metadata for URLs and citations...
Loading granular RAG chunks from rag_chunks.jsonl...
Loaded 950 chunks.
Loading structured data for numeric filtering...
Tokenizing documents and building BM25 index...
BM25 Index built successfully.

Searching for: 'What is the excess for maternity cover on the First Cover plan?'

🎯 Detected explicit request for plan: First Cover
Rank 1: First Cover (Irish Life Health)
URL: https://www.irishlifehealth.ie/mediafiles/pdfs/tables-of-cover/First-Cover-TOC.pdf
Scores -> Final: 0.6104 | Raw BM25: 19.2169 | Quantitative: 0.5

Evidence Preview:
What you're
covered for2
Irish Life Health dac is regulated by the Central Bank of Ireland.
First Cover
Effective from 1st January 2026
You should read this table of cover along with the Health Plans
membership handbook effective from January 2026, which you can
find on irishlifehealth.ie/more-info. The hospitals and treatment centres
covered on this plan are set out in List 1 in Part 12 of your Health Plans
m

# Hybrid

In [ ]:
import json
import faiss
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import difflib

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
RAG_CHUNKS_PATH = "rag_chunks.jsonl"
SUPERSET_PATH = "MASTER_STRUCTURED_SUPERSET_2026-1.jsonl"
URL_METADATA_PATH = "metadata (1).json"
INDEX_PATH = "faiss_multi_provider_index.bin"

# ==========================================
# 1. HELPER: TOKENIZER FOR BM25
# ==========================================
def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading FAISS embedding model...")
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

# ==========================================
# 3. LOAD METADATA
# ==========================================
print("Mapping metadata for URLs and citations...")
url_lookup = {}
human_plan_names = {}

try:
    with open(URL_METADATA_PATH, "r") as f:
        meta_data = json.load(f)
        for item in meta_data:
            doc_id = item.get("doc_id", "").lower()
            if doc_id:
                url_lookup[doc_id] = item.get("source_url", "No URL Available")
                human_plan_names[doc_id] = item.get("plan_name", "Unknown Plan")
except FileNotFoundError:
    print(f"Warning: {URL_METADATA_PATH} not found.")

def get_plan_details(doc_id):
    doc_id_lower = doc_id.lower()
    return (human_plan_names.get(doc_id_lower, doc_id.title()), url_lookup.get(doc_id_lower, "No URL Available"))

# ==========================================
# 4. LOAD RAG CHUNKS (For Both FAISS & BM25)
# ==========================================
documents_for_faiss = []
documents_for_bm25 = []
chunk_metadata = []

print(f"Loading granular RAG chunks from {RAG_CHUNKS_PATH}...")
try:
    with open(RAG_CHUNKS_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)

            doc_id = obj.get("doc_id", "")
            provider = obj.get("insurer", "Unknown Provider")
            plan_name = obj.get("plan_name", "")
            chunk_text = obj.get("text", "")

            enriched_chunk = f"Provider: {provider}. Plan: {plan_name}. Document text: {chunk_text}"
            documents_for_faiss.append(enriched_chunk)
            documents_for_bm25.append(chunk_text)

            chunk_metadata.append({
                "provider": provider,
                "doc_id": doc_id,
                "plan_name": plan_name,
                "chunk_text": chunk_text
            })
    print(f"Loaded {len(documents_for_faiss)} chunks.")
except FileNotFoundError:
    print(f"Error: {RAG_CHUNKS_PATH} not found.")

# ==========================================
# 5. LOAD STRUCTURED SUPERSET
# ==========================================
structured_data = {}
print("Loading structured data for numeric filtering...")
try:
    with open(SUPERSET_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)
            plan_key = obj.get("plan_name", "").lower()
            structured_data.setdefault(plan_key, []).append(obj)
except FileNotFoundError:
    print(f"Warning: {SUPERSET_PATH} not found.")

# ==========================================
# 6. BUILD HYBRID INDICES (FAISS & BM25)
# ==========================================
print("Building FAISS Index (Dense/Semantic)...")
if documents_for_faiss:
    embeddings = model.encode(documents_for_faiss, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

print("Building BM25 Index (Sparse/Lexical)...")
tokenized_corpus = [tokenize(doc) for doc in documents_for_bm25]
bm25 = BM25Okapi(tokenized_corpus)
print("Hybrid Indexes built successfully!")

# ==========================================
# 7. INSURANCE RISK ENGINE (ACTUARIAL SCORING)
# ==========================================
class InsuranceRiskEngine:
    def __init__(self, user_profile):
        self.profile = user_profile

    def evaluate_plan(self, plan_entries):
        if not plan_entries:
            return 0.5, True, {}

        # 1. HARD FILTERS (Dealbreakers)
        if self.profile.get("maternity_needed"):
            if not any(e.get("maternity_flag") for e in plan_entries):
                return 0.0, False, {"reason": "Maternity cover missing"}

        # 2. EXTRACT PLAN LIMITS
        inpatient_excesses = [float(e["excess_amount"]) for e in plan_entries if e.get("excess_amount") is not None and e.get("excess_type") == "inpatient"]
        outpatient_excesses = [float(e["excess_amount"]) for e in plan_entries if e.get("excess_amount") is not None and e.get("excess_type") == "outpatient"]

        avg_inpatient_excess = np.mean(inpatient_excesses) if inpatient_excesses else 150.0
        avg_outpatient_excess = np.mean(outpatient_excesses) if outpatient_excesses else 1.0

        # 3. CALCULATE EXPECTED OUT-OF-POCKET (OOP) COST
        projected_inpatient_cost = self.profile.get("inpatient_visits_yr", 0) * avg_inpatient_excess
        projected_outpatient_cost = self.profile.get("outpatient_visits_yr", 0) * avg_outpatient_excess

        total_projected_oop = projected_inpatient_cost + projected_outpatient_cost

        # 4. APPLY FINANCIAL RISK AVERSION PENALTY
        tolerance = self.profile.get("financial_tolerance", "medium")
        if tolerance == "low":
            risk_penalty_factor = 2.0
        elif tolerance == "high":
            risk_penalty_factor = 0.5
        else:
            risk_penalty_factor = 1.0

        adjusted_financial_burden = total_projected_oop * risk_penalty_factor

        # Normalize financial score: max burden (e.g., €2000+) = 0.0 score.
        financial_score = max(0.0, 1.0 - (adjusted_financial_burden / 2000.0))

        # 5. CHRONIC CONDITION MATCHING
        condition_score = 0.0
        conditions = self.profile.get("chronic_conditions", [])

        if conditions:
            plan_text_dump = str(plan_entries).lower()
            matched_conditions = sum(1 for condition in conditions if condition.lower() in plan_text_dump)
            condition_score = matched_conditions / len(conditions)
        else:
            condition_score = 1.0

        # 6. FINAL WEIGHTED ALGORITHM
        final_risk_score = (financial_score * 0.6) + (condition_score * 0.4)

        breakdown = {
            "projected_oop_cost": total_projected_oop,
            "financial_score": round(financial_score, 2),
            "condition_score": round(condition_score, 2)
        }

        return round(final_risk_score, 4), True, breakdown

# ==========================================
# 8. HYBRID SEARCH PIPELINE (RRF + RISK ENGINE)
# ==========================================
def hybrid_search(query, user_profile, k=60):
    query_lower = query.lower()
    mentioned_plan = None

    # Initialize Risk Engine for the current user
    engine = InsuranceRiskEngine(user_profile)

    # 1. Detect explicit plan requests
    unique_plans = set(meta["plan_name"] for meta in chunk_metadata)
    for known_plan in unique_plans:
        clean_known_plan = known_plan.replace("(Table of Cover)", "").strip().lower()
        if clean_known_plan and clean_known_plan in query_lower:
            mentioned_plan = known_plan
            print(f"🎯 Detected explicit request for plan: {mentioned_plan}")
            break

    # 2. Query Rewriting
    search_query = query
    if mentioned_plan:
        replace_target = mentioned_plan.replace("(Table of Cover)", "").strip()
        search_query = re.sub(replace_target, "", search_query, flags=re.IGNORECASE).strip()

    # --- A. FAISS (Semantic) Search ---
    prefixed_query = "Represent this sentence for searching relevant passages: " + search_query
    query_embedding = model.encode([prefixed_query], normalize_embeddings=True)
    query_embedding = np.array(query_embedding).astype("float32")
    faiss_scores, faiss_indices = index.search(query_embedding, k)

    # --- B. BM25 (Lexical) Search ---
    tokenized_query = tokenize(search_query)
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores)[::-1][:k]

    # --- C. Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}
    RRF_K = 60

    for rank, idx in enumerate(faiss_indices[0]):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1.0 / (RRF_K + rank + 1))

    for rank, idx in enumerate(bm25_indices):
        if bm25_scores[idx] > 0:
            rrf_scores[idx] = rrf_scores.get(idx, 0) + (1.0 / (RRF_K + rank + 1))

    # --- D. Group by Plan ---
    plan_chunks = {}
    provider_map = {}
    doc_id_map = {}

    for idx, rrf_score in rrf_scores.items():
        meta = chunk_metadata[idx]
        plan_name = meta["plan_name"]

        if plan_name not in plan_chunks:
            plan_chunks[plan_name] = []
            provider_map[plan_name] = meta["provider"]
            doc_id_map[plan_name] = meta["doc_id"]

        plan_chunks[plan_name].append({
            "text": meta["chunk_text"],
            "score": rrf_score
        })

    results = []

    for plan_name, chunks in plan_chunks.items():
        if mentioned_plan and plan_name != mentioned_plan:
            continue

        doc_id = doc_id_map[plan_name]

        chunks = sorted(chunks, key=lambda x: x["score"], reverse=True)
        best_rrf_score = chunks[0]["score"]

        # --- ROBUST FUZZY MAPPING ---
        # Strip out all the extra fluff words
        plan_key_clean = re.sub(r'\(.*?\)|table of cover|table of benefits|membership handbook|rules', '', plan_name, flags=re.IGNORECASE).strip().lower()

        # Use difflib to find the closest matching key in the structured data (80% similarity threshold)
        structured_keys = list(structured_data.keys())
        closest_matches = difflib.get_close_matches(plan_key_clean, structured_keys, n=1, cutoff=0.6)

        if closest_matches:
            matching_key = closest_matches[0]
            plan_entries = structured_data[matching_key]
            # print(f"✅ Mapped '{plan_name}' to structured key: '{matching_key}'")
        else:
            plan_entries = []
            # print(f"⚠️ No structured data found for: '{plan_name}'")

        # Compute Advanced Risk Score
        risk_score, passes_filter, breakdown = engine.evaluate_plan(plan_entries)
        if not passes_filter:
            continue

        # Normalize RRF Score
        normalized_rrf = min(best_rrf_score / 0.033, 1.0)

        # THE ULTIMATE SCORE COMBINATION:
        # 40% What the documents say (RAG/Hybrid) + 60% Cold, hard actuarial math (Risk Engine)
        final_score = (normalized_rrf * 0.4) + (risk_score * 0.6)

        human_name, source_url = get_plan_details(doc_id)
        if human_name == doc_id.title():
            human_name = plan_name

        combined_evidence = "\n\n--- NEXT CHUNK ---\n\n".join([c["text"] for c in chunks[:2]])

        results.append({
            "plan_name": human_name,
            "provider": provider_map[plan_name],
            "source_url": source_url,
            "final_score": round(final_score, 4),
            "hybrid_rrf_score": round(best_rrf_score, 4),
            "risk_score": round(risk_score, 4),
            "projected_oop": breakdown.get('projected_oop_cost', 0),
            "evidence": combined_evidence
        })

    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results[:5]

# ==========================================
# 9. EXECUTION / TEST
# ==========================================
if __name__ == "__main__":
    test_query = "What is the best plan for cardiac treatment?"

    # 1. Define the patient/user profile
    current_user = {
        "age": 45,
        "inpatient_visits_yr": 2,
        "outpatient_visits_yr": 4,
        "chronic_conditions": ["cardiac"],
        "maternity_needed": False,
        "financial_tolerance": "medium"
    }

    print(f"\nSearching for: '{test_query}'")
    print(f"Patient Profile: Expected {current_user['inpatient_visits_yr']} inpatient visits, Condition: {current_user['chronic_conditions']}\n")

    # 2. Run the search, passing both the query AND the user profile
    top_plans = hybrid_search(test_query, current_user)

    for i, r in enumerate(top_plans, 1):
        print(f"Rank {i}: {r['plan_name']} ({r['provider']})")
        print(f"URL: {r['source_url']}")
        print(f"Projected Out-of-Pocket Cost: €{r['projected_oop']}")
        print(f"Scores -> Final: {r['final_score']} | Actuarial Risk: {r['risk_score']} | Hybrid RRF: {r['hybrid_rrf_score']}")
        print(f"\nEvidence Preview:\n{r['evidence'][:300]}...\n") # Truncated for cleaner terminal output
        print("-" * 60)

Loading FAISS embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Mapping metadata for URLs and citations...
Loading granular RAG chunks from rag_chunks.jsonl...
Loaded 950 chunks.
Loading structured data for numeric filtering...
Building FAISS Index (Dense/Semantic)...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Building BM25 Index (Sparse/Lexical)...
Hybrid Indexes built successfully!

Searching for: 'What is the best plan for cardiac treatment?'
Patient Profile: Expected 2 inpatient visits, Condition: ['cardiac']

Rank 1: Hospital Plans Rules (Terms & Conditions) (Vhi)
URL: https://www.vhi.ie/pdf/products/Hospital%20Plans%20Rules_01Oct25_VHIMR2.pdf
Projected Out-of-Pocket Cost: €0
Scores -> Final: 0.6656 | Actuarial Risk: 0.5 | Hybrid RRF: 0.0302

Evidence Preview:
Co. Dublin (Medfit.ie), which is aimed at 
reducing the risk of a heart event.
You will have access to a specialist 
Cardiologist, cardiology Nurse, cardiac 
catheterisation laboratory and 
electrocardiogram (ECG) facilities.
We will pay the Benefit if:
1. You have an In-Patient Treatment cardiac 
 ...

------------------------------------------------------------
Rank 2: Tailored Health Plans Membership Handbook (Irish Life Health)
URL: https://www.irishlifehealth.ie/IrishLifeHealth/media/docs/ILH-Tailored-Health-Plans-Handbook-Ja

In [ ]:
import json
import faiss
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
RAG_CHUNKS_PATH = "rag_chunks.jsonl"
SUPERSET_PATH = "MASTER_STRUCTURED_SUPERSET_2026-1.jsonl"
URL_METADATA_PATH = "metadata (1).json"
INDEX_PATH = "faiss_multi_provider_index.bin"

# ==========================================
# 1. HELPER: TOKENIZER FOR BM25
# ==========================================
def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("Loading FAISS embedding model...")
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

# ==========================================
# 3. LOAD METADATA
# ==========================================
print("Mapping metadata for URLs and citations...")
url_lookup = {}
human_plan_names = {}

try:
    with open(URL_METADATA_PATH, "r") as f:
        meta_data = json.load(f)
        for item in meta_data:
            doc_id = item.get("doc_id", "").lower()
            if doc_id:
                url_lookup[doc_id] = item.get("source_url", "No URL Available")
                human_plan_names[doc_id] = item.get("plan_name", "Unknown Plan")
except FileNotFoundError:
    print(f"Warning: {URL_METADATA_PATH} not found.")

def get_plan_details(doc_id):
    doc_id_lower = doc_id.lower()
    return (human_plan_names.get(doc_id_lower, doc_id.title()), url_lookup.get(doc_id_lower, "No URL Available"))

# ==========================================
# 4. LOAD RAG CHUNKS (For Both FAISS & BM25)
# ==========================================
documents_for_faiss = []
documents_for_bm25 = []
chunk_metadata = []

print(f"Loading granular RAG chunks from {RAG_CHUNKS_PATH}...")
try:
    with open(RAG_CHUNKS_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)

            doc_id = obj.get("doc_id", "")
            provider = obj.get("insurer", "Unknown Provider")
            plan_name = obj.get("plan_name", "")
            chunk_text = obj.get("text", "")

            # FAISS needs enriched context
            enriched_chunk = f"Provider: {provider}. Plan: {plan_name}. Document text: {chunk_text}"
            documents_for_faiss.append(enriched_chunk)

            # BM25 works best on raw text
            documents_for_bm25.append(chunk_text)

            chunk_metadata.append({
                "provider": provider,
                "doc_id": doc_id,
                "plan_name": plan_name,
                "chunk_text": chunk_text
            })
    print(f"Loaded {len(documents_for_faiss)} chunks.")
except FileNotFoundError:
    print(f"Error: {RAG_CHUNKS_PATH} not found.")

# ==========================================
# 5. LOAD STRUCTURED SUPERSET
# ==========================================
structured_data = {}
print("Loading structured data for numeric filtering...")
try:
    with open(SUPERSET_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)
            plan_key = obj.get("plan_name", "").lower()
            structured_data.setdefault(plan_key, []).append(obj)
except FileNotFoundError:
    print(f"Warning: {SUPERSET_PATH} not found.")

# ==========================================
# 6. BUILD HYBRID INDICES (FAISS & BM25)
# ==========================================
print("Building FAISS Index (Dense/Semantic)...")
if documents_for_faiss:
    embeddings = model.encode(documents_for_faiss, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)

print("Building BM25 Index (Sparse/Lexical)...")
tokenized_corpus = [tokenize(doc) for doc in documents_for_bm25]
bm25 = BM25Okapi(tokenized_corpus)
print("Hybrid Indexes built successfully!")

# ==========================================
# 7. ADVANCED SCORING & FILTERING LOGIC
# ==========================================
def compute_structured_score(plan_key, query):
    plan_key_lower = plan_key.lower()
    matching_key = next((key for key in structured_data.keys() if key in plan_key_lower or plan_key_lower in key), None)

    if not matching_key: return 0.5, True

    plan_entries = structured_data[matching_key]
    query_lower = query.lower()

    if "maternity" in query_lower and not any(e.get("maternity_flag") for e in plan_entries): return 0.0, False
    if "psychiatric" in query_lower and not any(e.get("psychiatric_flag") for e in plan_entries): return 0.0, False
    if "fertility" in query_lower and not any(e.get("fertility_flag") for e in plan_entries): return 0.0, False

    excess_values = [float(e["excess_amount"]) for e in plan_entries if e.get("excess_amount") is not None]
    avg_excess = np.mean(excess_values) if excess_values else 0
    excess_score = 1 / (1 + avg_excess / 250)

    return (0.7 * excess_score) + 0.15, True

# ==========================================
# 8. HYBRID SEARCH PIPELINE (RRF)
# ==========================================
def hybrid_search(query, k=60):
    query_lower = query.lower()
    mentioned_plan = None

    # 1. Detect explicit plan requests
    unique_plans = set(meta["plan_name"] for meta in chunk_metadata)
    for known_plan in unique_plans:
        clean_known_plan = known_plan.replace("(Table of Cover)", "").strip().lower()
        if clean_known_plan and clean_known_plan in query_lower:
            mentioned_plan = known_plan
            print(f"🎯 Detected explicit request for plan: {mentioned_plan}")
            break

    # 2. Query Rewriting (Strip plan name so search focuses purely on the intent)
    search_query = query
    if mentioned_plan:
        replace_target = mentioned_plan.replace("(Table of Cover)", "").strip()
        search_query = re.sub(replace_target, "", search_query, flags=re.IGNORECASE).strip()

    # --- A. FAISS (Semantic) Search ---
    prefixed_query = "Represent this sentence for searching relevant passages: " + search_query
    query_embedding = model.encode([prefixed_query], normalize_embeddings=True)
    query_embedding = np.array(query_embedding).astype("float32")
    faiss_scores, faiss_indices = index.search(query_embedding, k)

    # --- B. BM25 (Lexical) Search ---
    tokenized_query = tokenize(search_query)
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores)[::-1][:k]

    # --- C. Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}
    RRF_K = 60 # Standard smoothing constant

    # Add FAISS Ranks
    for rank, idx in enumerate(faiss_indices[0]):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + (1.0 / (RRF_K + rank + 1))

    # Add BM25 Ranks
    for rank, idx in enumerate(bm25_indices):
        if bm25_scores[idx] > 0: # Only count if BM25 actually found a match
            rrf_scores[idx] = rrf_scores.get(idx, 0) + (1.0 / (RRF_K + rank + 1))

    # --- D. Group by Plan ---
    plan_chunks = {}
    provider_map = {}
    doc_id_map = {}

    for idx, rrf_score in rrf_scores.items():
        meta = chunk_metadata[idx]
        plan_name = meta["plan_name"]

        if plan_name not in plan_chunks:
            plan_chunks[plan_name] = []
            provider_map[plan_name] = meta["provider"]
            doc_id_map[plan_name] = meta["doc_id"]

        plan_chunks[plan_name].append({
            "text": meta["chunk_text"],
            "score": rrf_score
        })

    results = []

    for plan_name, chunks in plan_chunks.items():
        # EXPLICIT PLAN FILTER
        if mentioned_plan and plan_name != mentioned_plan:
            continue

        doc_id = doc_id_map[plan_name]

        # Sort chunks by their combined RRF Score
        chunks = sorted(chunks, key=lambda x: x["score"], reverse=True)
        best_rrf_score = chunks[0]["score"]

        # Compute Structured / Math Score
        structured_score, passes_filter = compute_structured_score(plan_name, query)
        if not passes_filter:
            continue

        # Normalize RRF Score for final weighting (max theoretical RRF is ~0.033)
        normalized_rrf = min(best_rrf_score / 0.033, 1.0)
        final_score = (normalized_rrf * 0.7) + (structured_score * 0.3)

        human_name, source_url = get_plan_details(doc_id)
        if human_name == doc_id.title():
            human_name = plan_name

        combined_evidence = "\n\n--- NEXT CHUNK ---\n\n".join([c["text"] for c in chunks[:2]])

        results.append({
            "plan_name": human_name,
            "provider": provider_map[plan_name],
            "source_url": source_url,
            "final_score": round(final_score, 4),
            "hybrid_rrf_score": round(best_rrf_score, 4),
            "structured_score": round(structured_score, 4),
            "evidence": combined_evidence
        })

    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results[:5]

# ==========================================
# 9. EXECUTION / TEST
# ==========================================
if __name__ == "__main__":
    test_query = "What is the best plan for cardiac treatment?"

    print(f"\nSearching for: '{test_query}'\n")
    top_plans = hybrid_search(test_query)

    for i, r in enumerate(top_plans, 1):
        print(f"Rank {i}: {r['plan_name']} ({r['provider']})")
        print(f"URL: {r['source_url']}")
        print(f"Scores -> Final: {r['final_score']} | Hybrid RRF: {r['hybrid_rrf_score']} | Quantitative: {r['structured_score']}")
        print(f"\nEvidence Preview:\n{r['evidence']}\n")
        print("-" * 60)

Loading FAISS embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Mapping metadata for URLs and citations...
Loading granular RAG chunks from rag_chunks.jsonl...
Loaded 950 chunks.
Loading structured data for numeric filtering...
Building FAISS Index (Dense/Semantic)...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Building BM25 Index (Sparse/Lexical)...
Hybrid Indexes built successfully!

Searching for: 'What is the best plan for cardiac treatment?'

Rank 1: Hospital Plans Rules (Terms & Conditions) (Vhi)
URL: https://www.vhi.ie/pdf/products/Hospital%20Plans%20Rules_01Oct25_VHIMR2.pdf
Scores -> Final: 0.7897 | Hybrid RRF: 0.0302 | Quantitative: 0.5

Evidence Preview:
Co. Dublin (Medfit.ie), which is aimed at 
reducing the risk of a heart event.
You will have access to a specialist 
Cardiologist, cardiology Nurse, cardiac 
catheterisation laboratory and 
electrocardiogram (ECG) facilities.
We will pay the Benefit if:
1. You have an In-Patient Treatment cardiac 
 admission; and
2. The programme is carried out at Medfit 
 Proactive Healthcare, Blackrock, Co. Dublin 
 (medfit.ie).
1. We will cover a Benefit when carried out on 
 an Out-Patient basis by a GP, Consultant, 
 Nurse or in a Approved Facility .
2. No Benefit is payable for shortfalls
 submitted against any other part of Your
 Plan.
 3. Re

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 10. LLM CONFIGURATION (Qwen-2.5-3B)
# ==========================================
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("\nLoading Expert Broker LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
llm_model.eval()

# ==========================================
# 11. FULL DOCUMENT RETRIEVAL HELPER
# ==========================================
def get_full_document_text(plan_name):
    """
    Simulates 'reading the entire PDF' by finding every chunk
    in the JSONL file that belongs to this specific plan.
    """
    full_text_list = []
    # We look up the doc_id from the global chunk_metadata we loaded earlier
    # to find all chunks sharing that ID.
    target_doc_id = next((m['doc_id'] for m in chunk_metadata if m['plan_name'] == plan_name), None)

    if not target_doc_id:
        return "Plan details not found in source documentation."

    for meta in chunk_metadata:
        if meta['doc_id'] == target_doc_id:
            full_text_list.append(meta['chunk_text'])

    return "\n\n".join(full_text_list)

# ==========================================
# 12. EXPERT RECOMMENDATION GENERATOR
# ==========================================
def generate_recommendation(user_query, user_profile, search_results):
    if not search_results:
        return "I'm sorry, I couldn't find a plan that meets your specific requirements."

    # 1. Take the top-ranked plan
    top_plan = search_results[0]

    # 2. Retrieve the 'Entire PDF' text
    print(f"📖 Reading the entire documentation for: {top_plan['plan_name']}...")
    entire_policy_text = get_full_document_text(top_plan['plan_name'])

    # 3. Build the prompt
    messages = [
        {"role": "system", "content": "You are an expert, empathetic Irish Health Insurance broker. You provide recommendations based on strict actuarial evidence and full policy documentation."},
        {"role": "user", "content": f"""
USER QUESTION: "{user_query}"

PATIENT PROFILE:
- Conditions: {', '.join(user_profile.get('chronic_conditions', []))}
- Expected Annual Inpatient Visits: {user_profile.get('inpatient_visits_yr', 0)}
- Financial Tolerance: {user_profile.get('financial_tolerance', 'medium')}

TOP SEARCH RESULT:
- Plan: {top_plan['plan_name']} ({top_plan['provider']})
- Search Relevance Score: {top_plan['final_score']}
- Key Snippets found: {top_plan['evidence']}

FULL POLICY DOCUMENTATION (Extracted from PDF):
{entire_policy_text[:15000]} # Using first 15k chars to stay within context limits

TASK:
Write a professional recommendation for the user.
1. Acknowledge their condition and medical needs.
2. Explain specifically how the benefits found in the 'FULL POLICY DOCUMENTATION' cover their needs.
3. Explicitly mention the specific hospitals or cardiac programs if mentioned in the text.
4. Conclude why this is the safest medical and financial choice for them.
"""}
    ]

    # 4. Generate with Qwen
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(llm_model.device)

    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=1000,
        temperature=0.3,
        do_sample=True
    )

    # Clean output
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

# ==========================================
# 13. UPDATED EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    test_query = "What is the best plan for cardiac treatment?"

    # Define the profile for the LLM to reason about
    current_user = {
        "chronic_conditions": ["cardiac"],
        "inpatient_visits_yr": 2,
        "financial_tolerance": "medium"
    }

    print(f"\nSearching for: '{test_query}'\n")

    # 1. Run your existing Hybrid Search
    top_plans = hybrid_search(test_query)

    # 2. Feed results into the LLM Expert Generator
    if top_plans:
        expert_report = generate_recommendation(test_query, current_user, top_plans)

        print("\n" + "="*60)
        print("🤖 EXPERT BROKER FINAL REPORT")
        print("="*60)
        print(expert_report)
        print("="*60)
        print(f"Source Document: {top_plans[0]['source_url']}")
    else:
        print("No results found.")


Loading Expert Broker LLM...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Searching for: 'What is the best plan for cardiac treatment?'

📖 Reading the entire documentation for: Hospital Plans Rules (Terms & Conditions)...

🤖 EXPERT BROKER FINAL REPORT
**Recommendation for Cardiac Treatment Plan**

Dear [Patient's Name],

Thank you for reaching out to us regarding your cardiac treatment needs. Based on the information provided and the policy details we've reviewed, I believe the **VHI Hospital Plans Rules (Terms & Conditions)**, particularly the **Medfit Cardiac Care Programme**, would be the most suitable option for your specific condition and financial tolerance.

### Acknowledgment of Condition and Needs

Your profile indicates a need for regular cardiac treatment, with expected annual inpatient visits of 2 times per year. Given your condition, it is crucial to have access to specialized cardiac care and rehabilitation programs to manage your health effectively and reduce the risk of future cardiac events.

### Coverage Explanation

#### Medfit Cardiac Ca

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 27.3 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import requests
import os
import io

# ==========================================
# 14. PDF DOWNLOADER & ANNOTATION ENGINE
# ==========================================
def process_policy_proofs(top_plan, chunk_metadata, output_folder="./annotated_reports"):
    """
    Fetches the PDF from the metadata URL, highlights evidence, and saves it.
    """
    # 1. Prepare Workspace
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    plan_name = top_plan['plan_name']
    source_url = top_plan.get('source_url')

    # Identify the doc_id to find the relevant chunks for highlighting
    # (Using your existing chunk_metadata list)
    doc_id = next((m['doc_id'] for m in chunk_metadata if m['plan_name'] == plan_name), None)

    if not source_url or source_url == "No URL Available":
        return "❌ Skip: No source URL found in metadata for this plan."

    print(f"🌐 Fetching original policy from: {source_url}...")

    try:
        # 2. Download the PDF
        response = requests.get(source_url, timeout=15)
        response.raise_for_status()
        pdf_stream = io.BytesIO(response.content)

        # 3. Open PDF with PyMuPDF
        doc = fitz.open(stream=pdf_stream, filetype="pdf")

        # 4. Highlight Evidence Chunks
        # We grab chunks belonging to this document that the search found
        # (Usually the top 3-5 chunks are the best 'proofs')
        proof_chunks = [m for m in chunk_metadata if m['doc_id'] == doc_id][:5]

        highlights_count = 0
        for chunk in proof_chunks:
            page_idx = chunk.get('page_start', 1) - 1
            if page_idx < 0 or page_idx >= len(doc): continue

            page = doc[page_idx]
            # Use a snippet of the text to find coordinates on the page
            search_text = chunk['chunk_text'][:150].strip()
            text_instances = page.search_for(search_text)

            for inst in text_instances:
                annot = page.add_highlight_annot(inst)
                annot.set_info(
                    title="Broker Evidence",
                    content=f"ACTUARIAL PROOF: This section confirms coverage for your specific medical needs."
                )
                annot.update()
                highlights_count += 1

        # 5. Save the annotated PDF
        safe_name = re.sub(r'[^\w\s-]', '', plan_name).strip().replace(' ', '_')
        output_path = os.path.join(output_folder, f"Annotated_{safe_name}.pdf")
        doc.save(output_path)
        doc.close()

        return f"✅ Success! {highlights_count} proofs highlighted. Download here: {output_path}"

    except Exception as e:
        return f"❌ Error processing PDF: {str(e)}"

# ==========================================
# 15. UPDATED FINAL EXECUTION
# ==========================================
if __name__ == "__main__":
    # ... (Your existing Hybrid Search & LLM Recommendation code) ...

    if 'top_plans' in locals() and top_plans:
        # 1. Generate the Deep-Reasoning Summary
        print("\n" + "="*60)
        print("🤖 GENERATING BROKER REPORT...")
        report = generate_recommendation(test_query, current_user, top_plans)
        print(report)

        # 2. Generate the Highlighted PDF Proof
        print("\n" + "="*60)
        print("🖍️  ANNOTATING PDF PROOFS...")
        top_match = top_plans[0]
        download_status = process_policy_proofs(top_match, chunk_metadata)
        print(download_status)
        print("="*60)


🤖 GENERATING BROKER REPORT...
📖 Reading the entire documentation for: Hospital Plans Rules (Terms & Conditions)...
**Recommendation for Cardiac Treatment Plan**

Dear [Patient's Name],

Thank you for reaching out to us regarding your cardiac treatment needs. Based on your profile, we understand that you have a condition requiring regular monitoring and potential treatment, with an expected annual inpatient visit of 2 times per year. Given your financial tolerance as medium, we believe the VHI Hospital Plans Rules (Terms & Conditions) (Vhi) offer a comprehensive and financially manageable solution.

### Understanding Your Needs

The VHI Hospital Plans Rules (Terms & Conditions) (Vhi) provide a robust framework for covering your cardiac treatment needs. Specifically, the plan offers several key benefits that align well with your medical requirements:

#### Inpatient Benefits
- **Specialist Cardiologist and Cardiology Nurse**: The plan ensures you have access to a specialist cardiologist

In [ ]:
import fitz  # PyMuPDF
import re
import os
import requests
import io

# ==========================================
# 14. CORRECTED RAG-TO-PDF ENGINE (AUTOMATED)
# ==========================================
def process_policy_proofs(top_plan_meta, chosen_rag_chunks, output_folder="./annotated_reports"):
    """
    Takes specific RAG chunks chosen by the LLM and
    annotates the PDF using the 'page_start' and 'text' metadata
    directly from those chunks.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    plan_name = top_plan_meta.get('plan_name', 'Health_Plan')
    source_url = top_plan_meta.get('source_url')

    if not source_url or source_url == "No URL Available":
        return {"status": "Error", "message": "❌ Skip: No source URL available."}

    print(f"🌐 Fetching original policy for: {plan_name}...")

    try:
        response = requests.get(source_url, timeout=30)
        response.raise_for_status()
        doc = fitz.open(stream=io.BytesIO(response.content), filetype="pdf")

        highlights_count = 0
        annotated_pages = set()

        # Iterate directly through the RAG chunks chosen by the LLM
        for chunk in chosen_rag_chunks:
            # --- AUTOMATED PAGE MAPPING ---
            page_start = chunk.get('page_start')
            chunk_id = chunk.get('chunk_id', 'Unknown')

            if page_start is None:
                print(f"⚠️ Skipping chunk {chunk_id}: Missing 'page_start'.")
                continue

            try:
                page_idx = int(page_start) - 1
            except ValueError:
                print(f"⚠️ Skipping chunk {chunk_id}: 'page_start' is not a number ({page_start}).")
                continue

            if page_idx < 0 or page_idx >= len(doc):
                print(f"⚠️ Page {page_start} out of range for chunk {chunk_id}.")
                continue

            page = doc[page_idx]

            # Use the text directly from the RAG chunk
            text_to_match = chunk.get('text')
            if not text_to_match:
                print(f"⚠️ Skipping chunk {chunk_id}: No text content.")
                continue

            # Find coordinates of the text (using snippet for robust matching)
            text_instances = page.search_for(text_to_match[:150])

            if not text_instances:
                print(f"⚠️ Could not find exact text match on page {page_start} for chunk {chunk_id[:20]}...")
                continue

            # --- APPLY VISUAL MARKUPS ---
            for inst in text_instances:
                highlight = page.add_highlight_annot(inst)

                # REQUIREMENT: Color coding (Yellow = Good Match)
                highlight.set_colors(stroke=(1, 1, 0)) # RGB Yellow

                # Attach reasoning from RAG chunk
                highlight.set_info(
                    title="Broker Evidence",
                    content=f"AI PROOF: {chunk.get('reasoning', 'Evidence found by LLM')}"
                )
                highlight.update()

                highlights_count += 1
                annotated_pages.add(page_start)

        # Save the annotated PDF
        safe_name = re.sub(r'[^\w\s-]', '', plan_name).strip().replace(' ', '_')
        output_path = os.path.join(output_folder, f"Annotated_{safe_name}.pdf")
        doc.save(output_path)
        doc.close()

        # REQUIREMENT: Return good page numbers
        return {
            "status": "Success",
            "message": f"✅ {highlights_count} proofs highlighted.",
            "pages": sorted(list(annotated_pages)),
            "download_path": output_path
        }

    except Exception as e:
        return {"status": "Error", "message": f"❌ Failed: {str(e)}"}
# ==========================================
# 15. UPDATED FINAL EXECUTION
# ==========================================
if __name__ == "__main__":
    # ... (Your existing Hybrid Search & LLM Recommendation code) ...

    if 'top_plans' in locals() and top_plans:


        # 2. Generate the Highlighted PDF Proof
        print("\n" + "="*60)
        print("🖍️  ANNOTATING PDF PROOFS...")
        top_match = top_plans[0]
        download_status = process_policy_proofs(top_match, chunk_metadata)
        print(download_status)
        print("="*60)


🖍️  ANNOTATING PDF PROOFS...
🌐 Fetching original policy for: Hospital Plans Rules (Terms & Conditions)...
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping chunk Unknown: Missing 'page_start'.
⚠️ Skipping